# KG1 V1244 CoT-safe — TREINO vigiado (Claude)
Rota A. **Run all.** Pré-req: Colab **A100 80GB** + Secret `HF_KEY` (escrita).
- Fixa `torch==2.10` (cu126) p/ casar o **wheel pronto do mamba** (~10s) — Colab veio com torch 2.11 (sem wheel).
- GPU-guard aborta cedo se VRAM insuficiente. Live-log: upload imediato + heartbeat + watchdog.
- `MODE='SMOKE'` (ensaio) ou `MODE='REAL'` (160 steps). Juíz de score = Notebook B (full947).

## 🧭 Como ler os logs (estilo *Use a Cabeça*)
Cada etapa imprime `[KG1-TEACH][ESTÁGIO][STATUS]`: **O que é / Por que importa / Como ler / Números-chave / Próxima ação**.
Status: `RUNNING/OK` 🟢 · `WATCH` 🟡 · `STOP/ABORT` 🔴 (watchdog matou).
Travamento: `KG1_WRAPPER_HEARTBEAT` a cada 45s (`last_output_age_s` crescendo = travando). Tudo sobe pro HF.
> 💡 `TRAIN_PULSE`=batida do coração. `SCORE_TRAJECTORY`=termômetro teacher-forced (não é score real; juíz=Notebook B).

In [1]:
import os, subprocess, sys
print('[1/5] clone repo branch', flush=True)
subprocess.run(['git','clone','--depth','1','--branch','claude/v1244-cot-safe','https://github.com/FELIPEACASTRO/KG1-NVIDIA.git','/content/kg1'], check=True)
os.chdir('/content/kg1')
print('[2/5] deps base', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','peft==0.19.1','accelerate==1.13.0','bitsandbytes','safetensors','huggingface_hub','hf_xet','einops','ninja'], check=False)
# PIN torch 2.10 (cu126): Colab veio 2.11 que NAO tem wheel do mamba 2.3.1 (max 2.10).
# --index-url garante build CUDA (PyPI default nao tem o linux CUDA). ANTES de importar torch.
print('[3/5] pin torch 2.10 cu126 (~2-3min; p/ casar o wheel do mamba)', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.10.0','--index-url','https://download.pytorch.org/whl/cu126'], check=False)
import torch
# BLINDAGEM: se o torch nao tiver CUDA, ABORTA (CPU quebraria mamba+treino).
assert torch.cuda.is_available() and torch.version.cuda, f'torch SEM CUDA apos pin (v={torch.__version__}, cuda={torch.version.cuda}). Reinicie e rode de novo.'
py=f"cp{sys.version_info.major}{sys.version_info.minor}"
tmm='.'.join(torch.__version__.split('+')[0].split('.')[:2])
cu='cu'+((torch.version.cuda or '12').split('.')[0])
abi='TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
print(f'[env] {py} torch{tmm} {cu} cxx11abi{abi} cuda_ok={torch.cuda.is_available()}', flush=True)
print('[4/5] mamba-ssm 2.3.1 (WHEEL PRONTO ~10s; fallback source se nao casar)', flush=True)
url=f"https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{tmm}cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
print('  tentando wheel:', url, flush=True)
if subprocess.run([sys.executable,'-m','pip','install','--no-deps',url]).returncode!=0:
    print('  >>> wheel nao casou -> compilando do source (~25min)', flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','mamba-ssm==2.3.1'], check=False)
print('[5/5] causal-conv1d 1.6.1 (source ~5min; sem wheel p/ cu12+torch2.x)', flush=True)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','causal-conv1d==1.6.1'], check=False)
# BLINDAGEM: import obrigatorio (deps quebradas abortam aqui, nao no treino).
try:
    import mamba_ssm, causal_conv1d
    print('IMPORT OK: mamba_ssm + causal_conv1d', flush=True)
except Exception as e:
    raise RuntimeError('FALHA import mamba_ssm/causal_conv1d apos install: '+str(e)[:200]+' -> reinicie o runtime e rode de novo.')
print('DEPS OK', flush=True)

[1/5] clone repo branch
[2/5] deps base
[3/5] pin torch 2.10 cu126 (~2-3min; p/ casar o wheel do mamba)
[env] cp312 torch2.10 cu12 cxx11abiTRUE cuda_ok=True
[4/5] mamba-ssm 2.3.1 (WHEEL PRONTO ~10s; fallback source se nao casar)
  tentando wheel: https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
[5/5] causal-conv1d 1.6.1 (source ~5min; sem wheel p/ cu12+torch2.x)
IMPORT OK: mamba_ssm + causal_conv1d
DEPS OK


In [2]:
from google.colab import userdata
import os
for k in ['HF_KEY','HF_TOKEN','HUGGINGFACE_TOKEN']:
    try:
        v=userdata.get(k)
        if v: os.environ['HF_TOKEN']=v; os.environ['HF_KEY']=v; break
    except Exception: pass
assert os.environ.get('HF_TOKEN'), 'Defina HF_KEY no Colab Secrets (com escrita)'
print('HF token OK', flush=True)

HF token OK


In [3]:
import torch
# GUARD DE GPU: NemotronH 30B (bf16) precisa ~60GB VRAM. A100 40GB -> OOM. Aborta CEDO e claro.
name=torch.cuda.get_device_name(0)
vram=torch.cuda.get_device_properties(0).total_memory/1e9
print(f'[GPU] {name} | VRAM total={vram:.0f}GB | usada={torch.cuda.memory_allocated(0)/1e9:.1f}GB', flush=True)
assert vram>=70, (f'VRAM={vram:.0f}GB < 70GB. O 30B precisa ~60GB -> esta A100 e 40GB. '
    'RECONECTE p/ A100 80GB (desconecte e reconecte ate vir 80GB). NAO treine em 40GB (OOM).')
print('[GPU] OK: VRAM suficiente p/ o 30B', flush=True)

[GPU] NVIDIA A100-SXM4-80GB | VRAM total=85GB | usada=0.0GB
[GPU] OK: VRAM suficiente p/ o 30B


In [4]:
import os, time
# ==========================================================================
#  ESCOLHA O MODO  (1 unico knob)
MODE = 'SMOKE'  # COMECE com 'SMOKE'. Troque p/ 'REAL' (160 steps) SO apos o GO do Claude.
# ==========================================================================
os.environ['DATA_FILE']='/content/kg1/artifacts/v1244_cot_safe/v1244_micro_consolidation_train.jsonl'
os.environ['VAL_FILE']='/content/kg1/artifacts/v1244_cot_safe/v1244_scorelive_evalset_170.jsonl'
os.environ['INIT_ADAPTER_REPO']='felipesp1983/kg1-recovered-v291-v290-checkpoint6-submit086'
os.environ['INIT_ADAPTER_REVISION']='f4134a6d223249d27be2f1c5d94ed59d118d1ce5'
os.environ['REQUIRE_INIT_ADAPTER']='1'
os.environ['LORA_R']='32'; os.environ['LORA_ALPHA']='32'
os.environ['BATCH_SIZE']='32'
os.environ['LEARNING_RATE']='5e-6'; os.environ['FINAL_LEARNING_RATE']='1e-6'
os.environ['BOXED_PAYLOAD_LOSS_WEIGHT']='1.0'
os.environ['REQUIRE_OFFSET_MASK']='1'
os.environ['OUTPUT_REPO']='felipesp1983/kg1-v1244-cot-candidate'
os.environ['UPLOAD_TO_HF']='1'
if MODE=='REAL':
    os.environ['MAX_STEPS']='160'; os.environ['NUM_EPOCHS']='6'
    os.environ['EVAL_EVERY_STEPS']='40'; os.environ['SAVE_EVERY_STEPS']='40'
    os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD']='0'
else:
    os.environ['MAX_STEPS']='8'; os.environ['NUM_EPOCHS']='1'
    os.environ['EVAL_EVERY_STEPS']='4'; os.environ['SAVE_EVERY_STEPS']='4'
    os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD']='1'
print('MODE=',MODE,'| MAX_STEPS=',os.environ['MAX_STEPS'],'NUM_EPOCHS=',os.environ['NUM_EPOCHS'],
      '| lr=',os.environ['LEARNING_RATE'],'->',os.environ['FINAL_LEARNING_RATE'],
      '| eval/save@',os.environ['EVAL_EVERY_STEPS'],'| live_log_require=',os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD'],flush=True)
print('Dataset=micro 979 | INIT=086 pinado | OUTPUT=',os.environ['OUTPUT_REPO'],flush=True)
print(('>>> ATENCAO: treino REAL (~2-3h). Confirme GO antes. <<<' if MODE=='REAL'
       else '>>> SMOKE: valida tudo + mede tempo/step. Apos OK do Claude, troque MODE=REAL. <<<'), flush=True)

MODE= SMOKE | MAX_STEPS= 8 NUM_EPOCHS= 1 | lr= 5e-6 -> 1e-6 | eval/save@ 4 | live_log_require= 1
Dataset=micro 979 | INIT=086 pinado | OUTPUT= felipesp1983/kg1-v1244-cot-candidate
>>> SMOKE: valida tudo + mede tempo/step. Apos OK do Claude, troque MODE=REAL. <<<


In [5]:
import subprocess, sys, os, time
os.chdir('/content/kg1')
os.environ['KG1_LIVE_LOG_HF_REPO']='felipesp1983/kg1-live-logs'
os.environ['KG1_LIVE_LOG_HF_REPO_TYPE']='dataset'
os.environ['RUN_ID']='v1244_train_'+time.strftime('%Y%m%d_%H%M%S')
os.environ.setdefault('KG1_REQUIRE_LIVE_LOG_UPLOAD','1')
os.environ.setdefault('KG1_WATCHDOG_STALE_SECONDS','2700')
os.environ.setdefault('KG1_LIVE_LOG_UPLOAD_EVERY','45')
print('LIVE RUN_ID=', os.environ['RUN_ID'], '-> HF:', os.environ['KG1_LIVE_LOG_HF_REPO']+'/colab/'+os.environ['RUN_ID'], flush=True)
print('   adapter ->', os.environ['OUTPUT_REPO']+'/runs/'+os.environ['RUN_ID']+'/{final,checkpoint-N}', flush=True)
r=subprocess.run([sys.executable,'scripts/kg1_colab_realtime_runner.py','--','python','scripts/hf_job_train_v90.py'])
print('RETURN_CODE=', r.returncode, flush=True)
print('Se RC=0 e adapter subiu -> rode NOTEBOOK B (CAND_RUN_ID='+os.environ['RUN_ID']+') p/ ACC real no full947.', flush=True)

LIVE RUN_ID= v1244_train_20260614_225748 -> HF: felipesp1983/kg1-live-logs/colab/v1244_train_20260614_225748
   adapter -> felipesp1983/kg1-v1244-cot-candidate/runs/v1244_train_20260614_225748/{final,checkpoint-N}
RETURN_CODE= 1
Se RC=0 e adapter subiu -> rode NOTEBOOK B (CAND_RUN_ID=v1244_train_20260614_225748) p/ ACC real no full947.
